# Job 50151317 — which sublayer writes the DD step? A null, and an informative one

hshunt asked which MLP layers run the Direct-Deduction steps. Nothing in the project could answer
that: the SAE pages (findings 19/21/29) train dictionaries on the **residual stream after** each
block, which is the sum of everything written up to that point, and activation patching (finding
27) splices whole residual cells. Both say the fact is in flight over L6-L10; neither says which
component put it there.

This run deletes **one write at a time**. These are pre-norm blocks, `x = x + attn(norm1(x))` then
`x = x + mlp(norm2(x))`, so zeroing `linear2`'s weight and bias makes block L's MLP write exactly
zero, and zeroing `out_proj` does the same for its attention. The skip connection is untouched, so
every later block still receives a well-formed residual stream. Scored as
`effect_retained = ablated counterfactual effect / intact effect`, the **same normalisation as the
head-ablation page**, so a sublayer number and a head number can be read on one axis.

**Headline: the DD step does not live in any one sublayer write, and this measurement cannot say
where it lives.** In the SOLVER, 20 of 24 writes are dispensable at the 0.90 line, most of them
completely (`effect_retained` 0.97-1.01). The four below it are the **read-in** (L0 attention,
0.021), the **write-out** (L11 MLP, 0.161), a partial at L10 attention (0.670) and L9 attention
just under the line at 0.896 — and each of them also damages the model's **clean** prediction in
proportion, which pre-registration (d) declared in advance makes them uninformative about
mechanism. Deleting a sublayer cannot detect a step computed redundantly in two places,
and the head page already showed exactly that redundancy.

**The one new positive result is about redundancy, and it comes from comparing the two
granularities on the same 350 cases.** At L0, the four attention heads individually retain
1.002 / **0.839** / 1.000 / 1.003 — no single head matters — yet deleting all four at once leaves
**0.021**. The same at L10: heads 0.975 / 0.994 / **0.843** / 1.001, whole block 0.670. **Head
ablation understates dependence because the heads inside a block back each other up.**


In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

S = os.environ["SCRATCH"] + "/spin_gnn"
SUB = f"{S}/sudoku_sublayer_2026-10-29"              # 50151317, this run
HEAD = f"{S}/sudoku_circuits_leakfree_2026-10-28"    # finding 27, the same cases
ARMS = {"A_leakfree_1000": "SOLVER (leak-free, 1.0000)",
        "C_leaked_0996": "LEAK-READER (leaked, 0.9960)"}
pd.set_option("display.width", 220)


def load(arm):
    d = pd.read_csv(f"{SUB}/{arm}/sublayer_ablation.csv")
    b = pd.read_csv(f"{SUB}/{arm}/behavior.csv")
    return d, b


SUBL = {a: load(a) for a in ARMS}
for a, lab in ARMS.items():
    d, b = SUBL[a]
    print(f"{lab:34s} n_cases {len(b):4d}  clean_correct {b.clean_correct.mean():.3f}  "
          f"corrupt_flipped {b.corrupt_flipped.mean():.3f}  clean margin {b.clean_logit_diff.mean():+.2f}")


## 1. The metric, and what the denominator actually is

Each row is normalised by **its own case's** intact effect, `base_effect`, the counterfactual
change in the logit difference; cases with `|base_effect| < 1e-6` are dropped (none were). The
cases are the same ones the patching and head-ablation pages used (`--n-instances 1200`,
`--n-cases 350`, same `--seed-start` as `sudoku2_circuits_leakfree.sbatch`), which is what lets
section 4 put heads and sublayers on one axis.

The denominator is solid in arm A and **soft in arm C**. The SOLVER gets the clean prompt right on
100% of cases and flips to DD's recomputed answer on **100%** of them, mean effect 31.22 logits.
The LEAK-READER is also clean-correct on 100% but flips on only **38.8%**, mean effect 12.54 —
which is itself the leak signature (a hint edit changes the deduced answer but not the adjacency
order it reads), and which leaves many arm-C rows as a ratio of two small numbers. Section 5
checks every arm-C claim against the median for that reason.

Read `effect_retained` as: **1.00 = that write is not needed for the flip, 0.00 = the flip is
gone.** Values above 1.00 mean the ablation *amplified* the effect.


In [ ]:
def profile(arm):
    d, _ = SUBL[arm]
    p = d.pivot_table(index="layer", columns="sublayer", values="effect_retained", aggfunc="mean")
    return p[["mlp", "attn"]]


tab = pd.concat({ARMS[a]: profile(a) for a in ARMS}, axis=1)
print("mean effect_retained per sublayer write")
print(tab.round(3).to_string())
for a, lab in ARMS.items():
    p = profile(a)
    print(f"\n{lab}: min MLP {p.mlp.min():.3f} at L{p.mlp.idxmin()}, "
          f"min attention {p.attn.min():.3f} at L{p.attn.idxmin()}, "
          f"max anywhere {p.max().max():.3f}")


## 2. Pre-registration (d), read first: is anything left of the model?

(d) was written as the gate on all the others: *if removing a sublayer also destroys the model's
clean prediction, then losing the counterfactual effect is not localisation, it is a broken
network.* The table below puts `effect_retained` next to the **clean** decision margin after the
same ablation.

Every write that loses effect loses clean margin by almost exactly the same fraction: L0 attention
retains 0.021 of the effect and **2%** of the clean margin, L11's MLP 0.161 and **15%**, L10
attention 0.670 and **65%**, L9 attention 0.896 and **88%**. That one-to-one tracking is the
signature of a broken forward pass rather than a deleted step. Nothing here is a dissociation —
there is no sublayer whose removal spares the clean prediction and kills the counterfactual, which
is what "this MLP runs the DD step" would have looked like.


In [ ]:
rows = []
for a, lab in ARMS.items():
    d, b = SUBL[a]
    clean = b.clean_logit_diff.mean()          # negative: margin for the TRUE colour
    g = d.groupby(["layer", "sublayer"])[["effect_retained", "ablated_clean_logit_diff"]].mean()
    g = g[g.effect_retained < 0.90]
    for (L, s), r in g.iterrows():
        rows.append({"arm": lab, "layer": L, "sublayer": s,
                     "effect_retained": r.effect_retained,
                     "clean_margin": -r.ablated_clean_logit_diff,
                     "clean_margin_intact": -clean,
                     "clean_frac_kept": r.ablated_clean_logit_diff / clean})
fail = pd.DataFrame(rows)
print("every write that loses effect (effect_retained < 0.90), with the sanity control")
print(fail.round(3).to_string(index=False))

for a, lab in ARMS.items():
    d, _ = SUBL[a]
    n = d.groupby(["layer", "sublayer"]).effect_retained.mean()
    print(f"{lab}: {int((n >= 0.90).sum())} of {len(n)} writes are fully dispensable")


## 3. The four pre-registrations, scored

From `scripts/slurm/sudoku2_sublayer.sbatch` lines 41-70, written before the job ran.

| | claim | verdict |
|---|---|---|
| (a) | at least one MLP retains <= +0.60, i.e. clearly less than the worst head's +0.839 | **CONFIRMED, and uninformative** |
| (b) | the load-bearing MLPs sit in L6-L10 | **FALSIFIED, into its own named falsifier** |
| (c) | the LEAK-READER's profile differs | **SPLIT** |
| (d) | sanity: the clean prediction survives | **holds at 20 of 24; fails at exactly the four that lose effect** |

* **(a)** L11's MLP retains 0.161. But L11 is the last block, and (d) disqualifies it: the clean
  margin goes 15.5 -> 2.4. The pre-registration's own named falsifier — "every MLP retains >= 0.839,
  which would mean the computation is spread over both sublayers with no single load-bearing write"
  — is what actually happened for **every MLP that is not the output write**.
* **(b)** Every MLP from L0 to L10 retains 0.97-1.01, with the 10th-90th percentile inside about
  +-0.02. The minimum sits at **L11**, which the sbatch named in advance as the falsifier that
  "would divorce the causal localisation from the representational one". The SAE's forcing-set
  features at L8 are therefore **not** matched by any causal dependence on L8's MLP.
* **(c)** The last MLP is shared (0.295 against the SOLVER's 0.161) but the read-in is not: the
  LEAK-READER's L0 attention retains **1.270** in the mean (0.823 median, section 5), against
  0.021 in the SOLVER. Several of its writes amplify (mean up to 1.288, median up to 1.16) — the
  sublayer-level echo of finding 35(a), where ablating L0H2/L9H1/L2H3 grows the leaked model's
  effect because they pull back toward the pre-edit (leak-derived) answer.
* **(d)** Held at 20 of 24, and it is the reason this page is a null rather than a localisation.


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True,
                         gridspec_kw={"height_ratios": [2, 1]})
MK = {"mlp": ("#2b6cb0", "o", "block L's MLP"), "attn": ("0.45", "s", "block L's attention")}
d, b = SUBL["A_leakfree_1000"]
clean = -b.clean_logit_diff.mean()
for s, (col, mk, lab) in MK.items():
    g = d[d.sublayer == s].groupby("layer")
    m = g.effect_retained.mean()
    lo, hi = g.effect_retained.quantile(0.1), g.effect_retained.quantile(0.9)
    axes[0].fill_between(m.index, lo, hi, color=col, alpha=0.18, lw=0)
    axes[0].plot(m.index, m, mk + "-", color=col, lw=2.2, label=lab)
    axes[1].plot(m.index, -g.ablated_clean_logit_diff.mean(), mk + "-", color=col, lw=2.2)
    bad = m[m < 0.90]
    axes[0].plot(bad.index, bad, "x", color="#c53030", ms=14, mew=3)
axes[0].axhline(1.0, color="k", ls="--", lw=1.5)
axes[0].axhline(0.839, color="#b7791f", ls=":", lw=2)
axes[0].text(0.3, 0.80, "0.839 = the worst SINGLE head (48 of them)", color="#b7791f")
axes[0].axvspan(5.6, 10.4, color="#f6e05e", alpha=0.25, lw=0)
axes[0].text(8.0, 0.55, "where the SAE finds\nthe forcing set", ha="center", color="#8a6d00")
axes[0].set_ylabel("fraction of the hint-edit\neffect retained")
axes[0].legend(loc="lower center", ncol=2)
axes[1].axhline(clean, color="k", ls="--", lw=1.5)
axes[1].axhline(0, color="#c53030", lw=2)
axes[1].set_ylabel("SANITY: clean margin")
axes[1].set_xlabel("block index L")
axes[1].text(1.0, 1.0, "0 = the clean prediction is now WRONG", color="#c53030")
fig.suptitle("SOLVER: 20 of 24 writes are dispensable; the four that are not also break the "
             "clean prediction", weight="bold")
fig.tight_layout()


## 4. Comparison to the previous best on the same eval

The previous — and only — sublayer-resolved result on these exact 350 counterfactuals is the
**per-head ablation** from finding 27 (`sudoku_circuits_leakfree_2026-10-28/*/head_ablation.csv`),
same normalisation, same cases, so this is a like-for-like comparison rather than a cross-pool one.

It is also the only place this run **adds** something. A head ablation deletes one of the four
heads in a block; this run deletes all four at once. Where those two disagree, the heads inside
that block are redundant with each other:

| block | worst single head | whole attention write | reading |
|---|---|---|---|
| L0 | 0.839 | **0.021** | four heads, each dispensable, jointly necessary |
| L10 | 0.843 | **0.670** | same, weaker |
| L1-L9, L11 | >= 0.896 | >= 0.896 | nothing to localise at either granularity |

So "no single head matters" (finding 27) and "the whole block matters" are both true at L0, and
the honest summary of the pair is **redundancy at the head level, not absence of a mechanism**.


In [ ]:
h = pd.read_csv(f"{HEAD}/A_leakfree_1000/head_ablation.csv")
hg = h.groupby(["layer", "head"]).effect_retained.mean()
sg = profile("A_leakfree_1000").attn

cmp = pd.DataFrame({"worst_single_head": hg.groupby("layer").min(),
                    "best_single_head": hg.groupby("layer").max(),
                    "whole_attn_write": sg})
cmp["head_minus_block"] = cmp.worst_single_head - cmp.whole_attn_write
print("heads vs the whole attention write, same 350 cases, same normalisation")
print(cmp.round(3).to_string())
print(f"\nheads below 0.95: {int((hg < 0.95).sum())} of {len(hg)}; "
      f"attention writes below 0.95: {int((sg < 0.95).sum())} of {len(sg)}")
print("L0 heads individually:", [round(v, 3) for v in hg.loc[0].values],
      "-> whole block", round(sg[0], 3))


## 5. Robustness: mean of ratios vs median of ratios

`effect_retained` is a mean of per-case ratios, so a case whose intact effect is small can move
the average a long way. Arm A is immune in practice (every case flips, mean effect 31.22) and its
mean and median agree to within 0.03 everywhere. Arm C is not: its L0 attention is **1.270 in the
mean and 0.823 in the median**, and its amplifications shrink from 1.288 to about 1.16.

What survives the median is the part the verdict rests on: **L0 attention is load-bearing in the
SOLVER (0.016) and not in the LEAK-READER (0.823)**, and the last MLP is load-bearing in both
(0.160 and 0.229). The "deleting it makes the edit *more* effective" phrasing is the fragile part
and should be read as "not needed, and noisy", not as a measured amplification.


In [ ]:
for a, lab in ARMS.items():
    d, _ = SUBL[a]
    g = d.groupby(["layer", "sublayer"]).effect_retained
    cmp2 = pd.DataFrame({"mean": g.mean(), "median": g.median()})
    cmp2["gap"] = cmp2["mean"] - cmp2["median"]
    worst = cmp2.gap.abs().sort_values(ascending=False).head(3)
    print(f"{lab}: largest mean-median gaps")
    print(cmp2.loc[worst.index].round(3).to_string(), "\n")
print("verdict-critical cells, median:")
for a, lab in ARMS.items():
    d, _ = SUBL[a]
    g = d.groupby(["layer", "sublayer"]).effect_retained.median()
    print(f"  {lab:34s} L0 attn {g[(0, 'attn')]:.3f}   L11 mlp {g[(11, 'mlp')]:.3f}")


## 6. Provenance, config, limitations

* **Job** `50151317`, `sud2sublayer`, COMPLETED 2026-10-29 10:51:05, elapsed **00:02:41**,
  `-p sganguli -G 1 --time=03:00:00 --cpus-per-task=4 --mem=32GB`.
* **Script** `scripts/slurm/sudoku2_sublayer.sbatch`; outputs
  `$SCRATCH/spin_gnn/sudoku_sublayer_2026-10-29/{A_leakfree_1000,C_leaked_0996}/`.
* **Checkpoints scored** (not retrained here): arm A the leak-free `L=12/ff=1024` 1.0000 model
  (finding 23, job `49705064`, seed 0), arm C the leaked 0.9960 record (finding 16's twin). Both
  `d_model=256`, `nhead=4`, 12 blocks, pre-norm — so 24 sublayer writes per arm.
* **Cases** 350 (arm A) and 250 (arm C) one-token hint-edit counterfactuals, `n_colors=4` so
  chance is 0.25; same `--seed-start` as the circuits run.
* **Deck** page 16 of `notebooks/sudoku_dd_verdict_2026-10-28/slides.pdf`.

**Limitations.** One seed, one checkpoint per arm, one intervention type, and a mean of per-case
ratios that section 5 shows is soft in arm C. The decisive one is
structural: **deleting one sublayer at a time cannot detect a step computed redundantly in two or
more places**, and section 4 shows that redundancy is real at the head level inside L0 and L10, so
the flat 1.00 across L1-L10 is consistent both with "those writes are irrelevant" and with "those
writes duplicate each other". Separating those needs joint ablations (pairs, or a whole window at
once), which this run does not do. Zeroing a weight is also a blunt instrument compared with mean
or resample ablation: it pushes the block off-distribution rather than removing its information.

**Correction recorded.** Page 17's LIMITATIONS line previously read "no single head and no single
MLP or attention write is needed for the effect". The second half is contradicted by this run's own
table and was fixed on 2026-10-31; page 16's entry (c) said the LEAK-READER "localises to the SAME
two sublayers", which is wrong about L0 attention (1.270 there, 0.021 in the SOLVER) and is now
scored SPLIT.
